In [ ]:
# Import Required Libraries
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_groq import ChatGroq
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from pinecone.grpc import PineconeGRPC as Pinecone
from pinecone import ServerlessSpec
from dotenv import load_dotenv
import os

In [ ]:
# Load Environment Variables
load_dotenv(override=True)

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")



In [ ]:
# Load PDF Documents
loader = DirectoryLoader(
    "../Data",
    glob="*.pdf",
    loader_cls=PyPDFLoader
)

documents = loader.load()
print(len(documents))

In [ ]:
# Split Documents into Chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

text_chunks = splitter.split_documents(documents)
print(len(text_chunks))

In [ ]:
# Load HuggingFace Embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [ ]:
# Connect to Pinecone
pc = Pinecone(api_key=PINECONE_API_KEY)
index_name = "medicalbot"

# Create Pinecone Index
# (Only if it doesn't exist)
existing_indexes = [index["name"] for index in pc.list_indexes()]
if index_name not in existing_indexes:
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )
print("Done")

In [ ]:
# Store Embeddings in Pinecone
docsearch = PineconeVectorStore.from_documents(
    documents=text_chunks,
    embedding=embeddings,
    index_name=index_name
)

In [ ]:
# Load Existing Pinecone Index
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embeddings
)

In [ ]:
# Create Retriever
retriever = docsearch.as_retriever(
    search_type="similarity",
    search_kwargs={"k":3}
)

In [ ]:
# Test Retriever
docs = retriever.invoke("What is Acne?")
print(docs[0].page_content)

In [ ]:
# Load Groq LLM
from dotenv import load_dotenv
import os
from langchain_groq import ChatGroq

load_dotenv(override=True)
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

llm = ChatGroq(
    groq_api_key=GROQ_API_KEY,
    model_name="llama-3.3-70b-versatile",
    temperature=0
)

response = llm.invoke("Hello")
print(response.content)

In [ ]:
# Define System Prompt
from langchain_core.prompts import ChatPromptTemplate

system_prompt = """
You are an expert medical AI assistant.

Your task is to answer the user's question as accurately as possible.

Instructions:
- First, use the retrieved context to answer the question.
- If the retrieved context does not contain enough information, use your own medical knowledge to provide a complete and accurate answer.
- Do not mention whether the answer came from the retrieved context or your own knowledge.
- Never mention the words "context", "retrieved documents", "PDF", or "general knowledge".
- Combine the available information naturally into a single, fluent answer.
- If the question is outside the medical domain, politely respond that you are designed to answer medical questions.
- Keep the answer clear, concise, and easy to understand.
- Use bullet points when appropriate.

Retrieved Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}")
    ]
)

In [ ]:
# Create Question Answer Chain
question_answer_chain = create_stuff_documents_chain(
    llm,
    prompt
)
# Create Retrieval-Augmented Generation (RAG) Chain
rag_chain = create_retrieval_chain(
    retriever,
    question_answer_chain
)

In [ ]:
# Test Query 
response = rag_chain.invoke(
    {
        "input":"What is Acromegaly and gigantism?"
    }
)

print(response["answer"])

In [ ]:
response = rag_chain.invoke(
    {
        "input":"What is Depression?"
    }
)

print(response["answer"])